# 检索增强生成 RAG

前两章，我们已经准备好了 RAG 的“检索”部分：

- 第 13 章：用向量嵌入按意思比较文字；
- 第 14 章：把资料切块、建立索引，可以快速找到最相关的段落。

这一章完成剩下的两步：把找到的段落放进消息里（**增强**），让大模型根据它们来回答（**生成**）。

我们会用两种方式来做：

1. **固定流程**：每个问题都先检索，再回答。
2. **作为工具**：把检索做成一个工具，交给学习助手，由大模型自己决定什么时候去查资料。

In [ ]:
import asyncio
import inspect
import json
import os
import re
import threading
import uuid
from abc import ABC, abstractmethod
from datetime import datetime, timedelta
from enum import Enum
from pathlib import Path
from typing import Annotated, Callable, List, get_args, get_origin, get_type_hints
from zoneinfo import ZoneInfo

import anthropic
import numpy as np
from anthropic.types import UserMessageParam, AssistantMessageParam, ToolParam, TextBlockParam, ToolUseBlockParam, ToolResultBlockParam
from mcp import Client
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam
from openai.types.chat import ChatCompletionToolParam, ChatCompletionMessageToolCallParam, ChatCompletionToolMessageParam

In [ ]:
def note(label: str, text: str):
    """Print one step of the agent's work, indented, so it stands apart from questions and answers."""
    lines = str(text).splitlines() or [""]
    print(f"  * {label}: {lines[0]}")
    for line in lines[1:]:
        print(f"    {line}")

In [ ]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"
    TOOL = "Tool"

In [ ]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.summary()}

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

In [ ]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

In [ ]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

In [ ]:
class ToolCall:

    def __init__(self, id: str, name: str, arguments: dict):
        self.id = id
        self.name = name
        self.arguments = arguments

    def to_dict(self) -> dict:
        return {"id": self.id, "name": self.name, "arguments": self.arguments}

    def __repr__(self) -> str:
        return f"{self.name}({json.dumps(self.arguments, ensure_ascii=False)})"

In [ ]:
class AssistantMessage(Message):

    def __init__(self, content: str, tool_calls: List[ToolCall] = None):
        super().__init__(MsgType.ASSISTANT)
        self.content = content
        self.tool_calls = tool_calls or []

    def summary(self) -> str:
        if self.tool_calls:
            return f"{self.content} [call: {self.tool_calls}]"
        return self.content

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "content": self.content,
                "tool_calls": [call.to_dict() for call in self.tool_calls]}

In [ ]:
class ToolResultMessage(Message):

    def __init__(self, tool_call_id: str, name: str, result: str):
        super().__init__(MsgType.TOOL)
        self.tool_call_id = tool_call_id
        self.name = name
        self.result = result

    def summary(self) -> str:
        return f"{self.name} -> {self.result}"

    def to_dict(self) -> dict:
        return {"type": self.msg_type.value, "tool_call_id": self.tool_call_id,
                "name": self.name, "content": self.result}

In [ ]:
def message_from_dict(data: dict) -> Message:
    if data["type"] == MsgType.ASSISTANT.value:
        return AssistantMessage(data["content"], [ToolCall(**c) for c in data.get("tool_calls", [])])
    if data["type"] == MsgType.TOOL.value:
        return ToolResultMessage(data["tool_call_id"], data["name"], data["content"])
    classes = {MsgType.SYSTEM.value: SystemMessage, MsgType.USER.value: UserMessage}
    return classes[data["type"]](data["content"])

In [ ]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

In [ ]:
class Tool:

    def __init__(self, name: str, description: str, parameters: dict, function: Callable):
        self.name = name
        self.description = description
        self.parameters = parameters  # JSON Schema of the arguments
        self.function = function

    def run(self, arguments: dict) -> str:
        return str(self.function(**arguments))

    def __call__(self, *args, **kwargs):
        return self.function(*args, **kwargs)

In [ ]:
JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean", list: "array", dict: "object"}


def tool(function: Callable) -> Tool:
    """Decorator: build a Tool from a function's name, docstring and type hints."""
    hints = get_type_hints(function, include_extras=True)
    properties, required = {}, []

    for name, param in inspect.signature(function).parameters.items():
        hint, description = hints.get(name, str), None
        if get_origin(hint) is Annotated:  # Annotated[int, "description"]
            hint, description = get_args(hint)[:2]
        properties[name] = {"type": JSON_TYPES.get(hint, "string")}
        if description:
            properties[name]["description"] = description
        if param.default is inspect.Parameter.empty:
            required.append(name)

    return Tool(
        name=function.__name__,
        description=inspect.getdoc(function) or "",
        parameters={"type": "object", "properties": properties, "required": required},
        function=function
    )

In [ ]:
@tool
def get_current_time(timezone: Annotated[str, "IANA timezone name, e.g. 'America/Toronto', 'Asia/Shanghai'"]) -> str:
    """Get the current local date and time in a given IANA timezone."""
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S %Z")


print(json.dumps(get_current_time.parameters, indent=2))
print(get_current_time("America/Toronto"))

In [ ]:
@tool
def add_days(date: Annotated[str, "Start date in YYYY-MM-DD format"],
             days: Annotated[int, "Number of days to add, negative to subtract"] = 1) -> str:
    """Add (or subtract, if negative) a number of days to a date. Returns the new date and its weekday."""
    result = datetime.strptime(date, "%Y-%m-%d") + timedelta(days=days)
    return result.strftime("%Y-%m-%d (%A)")


print(json.dumps(add_days.parameters, indent=2))
print(add_days("2026-11-10", 100))

In [ ]:
class ToolRegistry:

    def __init__(self, tools: List[Tool] = None):
        self.tools = {}
        for t in tools or []:
            self.register(t)

    def register(self, t: Tool) -> Tool:
        if t.name in self.tools:
            raise ValueError(f"duplicate tool name: '{t.name}'")
        self.tools[t.name] = t
        return t

    def list(self) -> List[Tool]:
        return list(self.tools.values())

    def run(self, call: ToolCall) -> str:
        t = self.tools.get(call.name)
        if t is None:
            return f"Error: unknown tool '{call.name}'"
        try:
            return t.run(call.arguments)
        except Exception as e:
            return f"Error: {e}"  # let the model see the error and try to fix it

In [ ]:
class MCPConnection:
    """A synchronous wrapper around the async mcp.Client (Part 6 will make the whole Agent async).

    The connection lives in one long-running coroutine on a background event loop;
    every call is submitted to that loop and we wait for its result.
    """

    def __init__(self, url: str):  # the URL of a remote MCP server
        self.loop = asyncio.new_event_loop()
        threading.Thread(target=self.loop.run_forever, daemon=True).start()
        ready = threading.Event()
        self.session = asyncio.run_coroutine_threadsafe(self._session(url, ready), self.loop)
        ready.wait()
        if self.session.done():  # connecting failed: raise the error here
            self.session.result()

    async def _session(self, url: str, ready: threading.Event):
        self.closed = asyncio.Event()
        try:
            async with Client(url) as client:
                self.client = client
                ready.set()
                await self.closed.wait()  # keep the connection open until close()
        finally:
            ready.set()

    def _run(self, coroutine):
        return asyncio.run_coroutine_threadsafe(coroutine, self.loop).result()

    def list_tools(self):
        return self._run(self.client.list_tools()).tools

    def call_tool(self, name: str, arguments: dict):
        return self._run(self.client.call_tool(name, arguments))

    def close(self):
        self.loop.call_soon_threadsafe(self.closed.set)
        self.session.result()
        self.loop.call_soon_threadsafe(self.loop.stop)

In [ ]:
class MCPTool(Tool):
    """A tool that lives on an MCP server: same interface as Tool, but run() goes over MCP."""

    def __init__(self, connection: MCPConnection, mcp_tool):
        super().__init__(mcp_tool.name, mcp_tool.description or "", mcp_tool.input_schema, function=None)
        self.connection = connection

    def run(self, arguments: dict) -> str:
        result = self.connection.call_tool(self.name, arguments)
        text = "\n".join(block.text for block in result.content if block.type == "text")
        if result.is_error and not text.startswith("Error"):
            text = f"Error: {text}"  # same convention as ToolRegistry.run()
        return text


def load_mcp_tools(connection: MCPConnection) -> List[Tool]:
    return [MCPTool(connection, t) for t in connection.list_tools()]

In [ ]:
class Memory:
    SUMMARIZE_SYSTEM_MSG = "You are a conversation summary assistant."
    SUMMARIZE_USER_MSG = "Please condense the following conversation into a brief summary: "
    SUMMARY_PREFIX = "Historical Summary: "

    def __init__(self, provider, max_tokens=1000, max_messages=3, keep_messages=1, memory_id=None):
        self.provider = provider
        self.id = memory_id or uuid.uuid4().hex[:8]
        self.messages: List[Message] = []
        self.max_tokens = max_tokens
        self.max_messages = max_messages
        self.keep_messages = keep_messages

    def add(self, *messages: Message):
        # add a whole turn at once, so it is summarized (at most) once per turn
        self.messages.extend(messages)
        if len(self.messages) > self.max_messages or self.token_count() > self.max_tokens:
            self.summarize()

    def reset(self):
        self.messages = []

    def history(self, messages: List[Message] = None) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in (messages or self.messages)
        )

    def summarize(self):
        if self.keep_messages >= len(self.messages):
            return

        split = len(self.messages) - self.keep_messages
        # never separate a tool call from its result: keep from the last UserMessage onwards
        while split > 0 and not isinstance(self.messages[split], UserMessage):
            split -= 1
        if split == 0:
            return
        to_summarize = self.messages[:split]
        to_keep = self.messages[split:]

        request = UserMessage(Memory.SUMMARIZE_USER_MSG + "\n" + self.history(to_summarize))
        summary = self.provider.call(SystemMessage(Memory.SUMMARIZE_SYSTEM_MSG), [request])
        self.messages = [UserMessage(Memory.SUMMARY_PREFIX + summary.summary())] + to_keep
        note("summary", self.history())

    def token_count(self):
        # rough estimate: counts characters, not real tokens
        return sum(len(msg.summary() or "") for msg in self.messages)

    def to_dict(self) -> dict:
        return {"id": self.id, "messages": [msg.to_dict() for msg in self.messages]}

    @classmethod
    def from_dict(cls, data: dict, provider) -> "Memory":
        memory = cls(provider, memory_id=data["id"])
        memory.messages = [message_from_dict(m) for m in data["messages"]]
        return memory

In [ ]:
class MemoryStore:

    def __init__(self, root="memories"):
        self.root = Path(root)
        self.root.mkdir(exist_ok=True)

    def save(self, memory: Memory):
        path = self.root / f"{memory.id}.json"
        path.write_text(json.dumps(memory.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")

    def load(self, memory_id: str, provider) -> Memory:
        path = self.root / f"{memory_id}.json"
        if not path.exists():
            return None
        return Memory.from_dict(json.loads(path.read_text(encoding="utf-8")), provider)

In [ ]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        pass

In [ ]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ChatCompletionToolParam(type="function", function={
                "name": tool.name, "description": tool.description, "parameters": tool.parameters})
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                param = ChatCompletionAssistantMessageParam(role="assistant", content=msg.content)
                if msg.tool_calls:
                    param["tool_calls"] = [
                        ChatCompletionMessageToolCallParam(id=call.id, type="function", function={
                            "name": call.name, "arguments": json.dumps(call.arguments)})
                        for call in msg.tool_calls
                    ]
                schema.append(param)
            elif isinstance(msg, ToolResultMessage):
                # OpenAI: one "tool" message per tool result
                schema.append(ChatCompletionToolMessageParam(role="tool", tool_call_id=msg.tool_call_id, content=msg.result))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        tool_calls = [
            ToolCall(call.id, call.function.name, json.loads(call.function.arguments))
            for call in (message.tool_calls or [])
        ]
        return AssistantMessage(message.content or "", tool_calls)

In [ ]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], tools: List[Tool] = None, model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        if tools:
            kwargs["tools"] = self._tool_schema(tools)
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _tool_schema(tools: List[Tool]):
        return [
            ToolParam(name=tool.name, description=tool.description, input_schema=tool.parameters)
            for tool in tools
        ]

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                content = [TextBlockParam(type="text", text=msg.content)] if msg.content else []
                content += [
                    ToolUseBlockParam(type="tool_use", id=call.id, name=call.name, input=call.arguments)
                    for call in msg.tool_calls
                ]
                schema.append(AssistantMessageParam(role="assistant", content=content))
            elif isinstance(msg, ToolResultMessage):
                # Claude: all tool results of one turn go into a single "user" message
                block = ToolResultBlockParam(type="tool_result", tool_use_id=msg.tool_call_id, content=msg.result)
                if schema and schema[-1]["role"] == "user" and isinstance(schema[-1]["content"], list):
                    schema[-1]["content"].append(block)
                else:
                    schema.append(UserMessageParam(role="user", content=[block]))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        tool_calls = [
            ToolCall(block.id, block.name, block.input)
            for block in response.content if block.type == "tool_use"
        ]
        return AssistantMessage(text, tool_calls)

In [ ]:
class EmbeddingProvider(ABC):
    model: str = ""  # name of the embedding model, saved together with an index

    @abstractmethod
    def embed(self, texts: List[str]) -> np.ndarray:
        """Return one vector per text, shape (len(texts), dimension)."""
        pass

In [ ]:
class OpenAIEmbedding(EmbeddingProvider):

    def __init__(self, model: str = "text-embedding-3-small"):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.model = model

    def embed(self, texts: List[str]) -> np.ndarray:
        response = self.client.embeddings.create(model=self.model, input=texts)
        data = sorted(response.data, key=lambda d: d.index)  # one embedding per input, in order
        return np.array([d.embedding for d in data])

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Cosine similarity between every row of a and every row of b, shape (len(a), len(b))."""
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    return a @ b.T

In [ ]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage, registry: ToolRegistry = None, max_steps: int = 5):
        self.provider = provider
        self.system = system
        self.registry = registry or ToolRegistry()
        self.max_steps = max_steps
        self.store = MemoryStore()
        self.memory = Memory(provider)

    def run(self, message: str, system: SystemMessage = None) -> AssistantMessage:
        system = system or self.system
        tools = self.registry.list()
        turn = [UserMessage(message)]

        response = self.provider.call(system, self.memory.messages + turn, tools)
        turn.append(response)

        step = 0
        while response.tool_calls and step < self.max_steps:
            step += 1
            for call in response.tool_calls:
                result = self.registry.run(call)
                note(f"tool[{step}]", f"{call} -> {result}")
                turn.append(ToolResultMessage(call.id, call.name, result))
            response = self.provider.call(system, self.memory.messages + turn, tools)
            turn.append(response)

        if response.tool_calls:
            response.tool_calls = []
            response.content = response.content or f"(stopped after {self.max_steps} steps)"

        self.memory.add(*turn)
        self.store.save(self.memory)
        return response

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None  # preset questions, or None to type
        note("memory", f"#{self.memory.id}")
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            if message.strip().lower() == "/new":
                self.memory = Memory(self.provider)
                note("new memory", f"#{self.memory.id}")
                continue

            if message.startswith("#"):
                memory = self.store.load(message[1:].strip(), self.provider)
                if memory is None:
                    note("memory", f"{message} not found")
                else:
                    self.memory = memory
                    note(f"loaded memory #{memory.id}", memory.history())
                continue

            response = self.run(message, system)
            print(f"AI: {response.content}")

In [ ]:
class Chunk:

    def __init__(self, text: str, source: str):
        self.text = text
        self.source = source  # which document it comes from, used for citations in chapter 15

    def to_dict(self) -> dict:
        return {"text": self.text, "source": self.source}


def load_documents(folder: str) -> List[tuple]:
    """Return (source, text) for every Markdown file in the folder."""
    return [(path.name, path.read_text(encoding="utf-8")) for path in sorted(Path(folder).glob("*.md"))]

In [ ]:
def chunk_fixed(text: str, size: int = 200, overlap: int = 50) -> List[str]:
    """Cut the text every `size` characters; neighbouring chunks share `overlap` characters."""
    step = size - overlap
    return [text[start:start + size] for start in range(0, max(len(text) - overlap, 1), step)]

In [ ]:
def chunk_by_heading(text: str, source: str, max_size: int = 500) -> List[Chunk]:
    """One chunk per "## " section, prefixed with its headings; long sections are cut again."""
    chunks, title, heading, lines = [], "", "", []

    def flush():
        body = "\n".join(lines).strip()
        if body:
            prefix = " > ".join(h for h in (title, heading) if h)
            pieces = chunk_fixed(body, max_size, overlap=50) if len(body) > max_size else [body]
            chunks.extend(Chunk(f"{prefix}\n{piece}", source) for piece in pieces)
        lines.clear()

    for line in text.splitlines():
        if line.startswith("# "):
            flush()
            title, heading = line[2:].strip(), ""
        elif line.startswith("## "):
            flush()
            heading = line[3:].strip()
        else:
            lines.append(line)
    flush()
    return chunks

## 给搜索加一个门槛

`VectorIndex.search` 多了一个参数 `min_score`：相似度低于它的结果，直接丢掉。

为什么需要它？`search` 总是会返回最相似的 `top_k` 块，哪怕它们和问题根本没有关系。如果小明问的问题知识库里没有，这些“最相似”的块其实都不相关，交给大模型只会误导它。

In [ ]:
class VectorIndex:

    def __init__(self, embedder: EmbeddingProvider):
        self.embedder = embedder
        self.chunks: List[Chunk] = []
        self.vectors = None  # shape (len(chunks), dimension)

    def add(self, chunks: List[Chunk], batch_size: int = 100):
        batches = [chunks[i:i + batch_size] for i in range(0, len(chunks), batch_size)]
        vectors = np.vstack([self.embedder.embed([c.text for c in batch]) for batch in batches])
        self.vectors = vectors if self.vectors is None else np.vstack([self.vectors, vectors])
        self.chunks += chunks

    def search(self, query: str, top_k: int = 3, min_score: float = 0.0) -> List[tuple]:
        if not self.chunks:
            return []
        scores = cosine_similarity(self.embedder.embed([query]), self.vectors)[0]
        best = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[i], float(scores[i])) for i in best if scores[i] >= min_score]

    def save(self, folder: str):
        folder = Path(folder)
        folder.mkdir(exist_ok=True)
        np.save(folder / "vectors.npy", self.vectors)
        data = {"model": self.embedder.model, "chunks": [c.to_dict() for c in self.chunks]}
        (folder / "chunks.json").write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

    @classmethod
    def load(cls, folder: str, embedder: EmbeddingProvider) -> "VectorIndex":
        folder = Path(folder)
        data = json.loads((folder / "chunks.json").read_text(encoding="utf-8"))
        if data["model"] != embedder.model:  # vectors of different models can not be compared
            raise ValueError(f"index was built with '{data['model']}', not '{embedder.model}'")
        index = cls(embedder)
        index.chunks = [Chunk(**c) for c in data["chunks"]]
        index.vectors = np.load(folder / "vectors.npy")
        return index

这一章用的知识库，是第 14 章写进 `knowledge` 文件夹的三篇文章。如果你还没有运行过第 14 章，请先运行它。

In [ ]:
# the knowledge base was written into knowledge/ in chapter 14
assert Path("knowledge/mcp.md").exists(), "please run chapter 14 first"

我们再加一篇文章，介绍本书的学习助手本身：有哪些命令、记忆保存在哪里、工具调用最多几步等。这些内容大模型肯定不知道，正好用来检验 RAG 的效果。

In [ ]:
%%writefile knowledge/this-book.md
# 本书的 Agent

## 命令
在本书的 Agent 对话中，输入 /quit 退出，输入 /new 开始一段新的记忆，输入 # 加记忆 ID（例如 #a1b2c3d4）加载以前保存的记忆。

## 记忆的保存
每一轮对话结束后，Agent 会自动把记忆保存到 memories 目录下，文件名是记忆 ID 加上 .json 后缀。记忆 ID 是 8 位十六进制字符。

## 工具调用的步数
Agent 每轮对话最多执行 5 步工具调用（max_steps=5）。超过以后，Agent 会停止调用工具，并回复 "(stopped after 5 steps)"。

## MCP 连接
第 12 章的 Agent 通过 MCPConnection 连接远程 MCP 服务器，演示使用的是 DeepWiki 服务器（https://mcp.deepwiki.com/mcp），它可以回答关于 GitHub 公开仓库的问题。

重新建立索引并保存。

In [ ]:
embedder = OpenAIEmbedding()
index = VectorIndex(embedder)
for source, text in load_documents("knowledge"):
    index.add(chunk_by_heading(text, source))
index.save("index")
print(len(index.chunks), "chunks from", sorted({c.source for c in index.chunks}))

## 门槛定多少？

`min_score` 设多少合适，和嵌入模型有关，没法事先给出一个标准答案。最实际的办法是**先看看分数**：找一个知识库里有答案的问题，再找一个完全无关的问题，比较它们的最高分。

门槛应该定在两者之间：比相关问题的分数低，比无关问题的分数高。这里先设为 `0.3`，请根据你自己运行的结果调整。

In [ ]:
# look at the raw scores first, to choose a threshold
for question in ["MCP 有哪些传输方式？", "蒙特利尔一月有多冷？"]:
    print(question, [round(score, 2) for _, score in index.search(question, top_k=3)])

MIN_SCORE = 0.3  # relevant passages score clearly higher than unrelated ones: tune it with the numbers above

## 方式一：固定流程

`rag_answer` 把 RAG 的三步写成一个函数：

1. **检索**：在索引里找出最相关的几块。如果一块都没找到，直接回答“知识库里没有”，连大模型都不用调用。
2. **增强**：把找到的块连同出处拼在一起，放进提示词里。
3. **生成**：调用大模型，得到回答。

提示词 `RAG_PROMPT` 是关键，它对大模型提了三个要求：

- **只根据给出的资料回答**，不要自己编；
- **注明出处**，比如 `[mcp.md]`，方便小明回去查原文；
- **资料里没有答案时，就说不知道**。

后两条很重要。大模型有个毛病：不知道的事情，也常常一本正经地编一个答案出来，这叫作**幻觉**（hallucination）。要求它注明出处、允许它说不知道，能大大减少这种情况。

In [ ]:
RAG_PROMPT = """Answer the question using only the context below.
Cite the sources you used, like [mcp.md].
If the context does not contain the answer, say that you don't know.

Context:
{context}

Question: {question}"""


def rag_answer(question: str, index: VectorIndex, provider: LLMProvider, top_k: int = 3) -> str:
    results = index.search(question, top_k, min_score=MIN_SCORE)                  # 1. retrieve
    if not results:
        return "I couldn't find anything about this in the knowledge base."
    context = "\n\n".join(f"[{chunk.source}]\n{chunk.text}" for chunk, _ in results)
    prompt = RAG_PROMPT.format(context=context, question=question)                 # 2. augment
    return provider.call(system_message, [UserMessage(prompt)]).content            # 3. generate

用同一个关于本书学习助手的问题，比较一下不用 RAG 和用了 RAG 的回答。不用 RAG 时，大模型只能靠猜；用了 RAG，它能说出 `#` 加记忆 ID 这个具体的命令，还会注明出处。

In [ ]:
provider = OpenAIProvider()
question = "在本书的 Agent 里，怎样切换回以前的一段对话？"

print("Without RAG:\n", provider.call(system_message, [UserMessage(question)]).content)
print("\nWith RAG:\n", rag_answer(question, index, provider))

再试两个问题：一个知识库里有，一个知识库里没有。第二个问题因为找不到足够相似的资料，会直接得到“知识库里没有”的回答。

In [ ]:
for question in ["MCP 有哪些传输方式？", "蒙特利尔一月有多冷？"]:  # the second one is not in the knowledge base
    print("Q:", question)
    print(rag_answer(question, index, provider), "\n")

## 方式二：把检索做成工具

固定流程有个缺点：**每个问题都会去检索**。小明问“今天星期几”，也要先在知识库里查一遍，既浪费，又可能找来不相关的资料干扰回答。

更灵活的做法是：把检索做成一个工具 `search_knowledge`，和其他工具一起交给学习助手。大模型看到问题后，**自己判断**需不需要查资料、用什么关键词去查，甚至可以查好几次。

有了第 11 章的 `@tool`，写这个工具很简单。注意它的描述：写清楚知识库里有哪些内容，大模型才知道什么样的问题该来这里查。

In [ ]:
@tool
def search_knowledge(query: Annotated[str, "What to look up, as a short question or keywords"]) -> str:
    """Search the knowledge base about this book: tool calling, MCP, memory and the book's own agent.
    Returns the most relevant passages together with their source file."""
    results = index.search(query, top_k=3, min_score=MIN_SCORE)
    if not results:
        return "No relevant passages found in the knowledge base."
    return "\n\n".join(f"[{chunk.source}] (score {score:.2f})\n{chunk.text}" for chunk, score in results)


print(search_knowledge("max_steps"))

给学习助手换一个系统消息，告诉它：遇到关于本书的问题，先用 `search_knowledge` 查资料，并注明出处。工具箱里放两个工具：`add_days` 和 `search_knowledge`。

`Agent` 本身一行都不用改。对它来说，`search_knowledge` 只是工具箱里的又一个工具。

先用 `agent.run` 问一个问题，只看最后的回答：

In [ ]:
rag_system = SystemMessage(
    "You're an AI assistant. For questions about this book or its agent, "
    "use search_knowledge first and cite the sources you used, like [mcp.md]."
)
registry = ToolRegistry([add_days, search_knowledge])
agent = Agent(provider, rag_system, registry=registry)

response = agent.run("本书的 Agent 一轮对话最多能调用几次工具？")
print("AI:", response.content)

再用 `chat` 连续问三个问题，观察学习助手分别用了哪个工具：

- 第一个是关于本书的问题，应该用 `search_knowledge`；
- 第二个是日期问题，应该用 `add_days`；
- 第三个知识库里没有，看看它怎么处理。

In [ ]:
agent.chat([
    "本书的 Agent 用什么命令开始一段新的记忆？",  # in the knowledge base: search_knowledge
    "2026 年 12 月 1 日之后的第 30 天是星期几？",  # a date question: add_days
    "蒙特利尔一月有多冷？",                        # not in the knowledge base
])
# or call agent.chat() without questions to type your own

> 把检索交给大模型自己决定，这种做法有时被称为 **Agentic RAG**。现在很多智能体产品都是这样做的，比如 Claude Code 在回答关于代码的问题时，会自己决定去搜索哪些文件。

## 动手试试

- 把 `RAG_PROMPT` 里“If the context does not contain the answer, say that you don't know.”这一句删掉，把 `MIN_SCORE` 设为 0，再问“蒙特利尔一月有多冷？”，看看回答有什么变化。
- 在 `knowledge/this-book.md` 里补充一条内容，比如第 11 章的工具箱，重新建立索引，然后问学习助手相关的问题。
- 把 `search_knowledge` 的描述改得很简单，比如“Search.”，看看学习助手还会不会在该用的时候用它。

## 小结

- RAG 的完整流程：**检索**相关段落，**增强**提示词，**生成**回答。
- 用 `min_score` 过滤掉不相关的结果。门槛要根据实际的分数来定。
- 好的 RAG 提示词要求大模型：只根据资料回答、注明出处、不知道就说不知道，以减少**幻觉**。
- RAG 可以是**固定流程**，也可以做成**工具**，让大模型自己决定什么时候查、查什么。
- 检索工具只是工具箱里的又一个工具，`Agent` 不用改。

现在，学习助手能根据知识库回答问题了。不过，它还记不住小明这个人：每次开始新的对话，它都不知道小明在学什么、喜欢怎样的讲解方式。下一章，我们用同样的向量检索技术，给它加上**长期记忆**。